In [1]:
import os
import sys
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from skimage.color import rgb2lab, lab2rgb
import torch

sys.path.append(os.path.expanduser("~/colorizer_gan/notebooks"))
from model import Generator

In [3]:
MODEL_PATH  = os.path.expanduser("~/colorizer_gan/models/checkpoint_epoch_100.pth")
VAL_PATH    = "/mnt/hdd/colorizer_gan/datasets/coco/val/val2017"
OUTPUT_PATH = os.path.expanduser("~/colorizer_gan/outputs/showcase")
IMAGE_SIZE  = 256
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load generator
def load_generator(model_path, device):
    G          = Generator().to(device)
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    G.load_state_dict(checkpoint["G_state"])
    G.eval()
    print(f"Generator loaded — epoch {checkpoint['epoch']}")
    return G

G = load_generator(MODEL_PATH, DEVICE)
print(f"Device: {DEVICE}")

Generator loaded — epoch 100
Device: cuda


In [4]:
def colorize(image_path, G, device, image_size=256):
    img    = Image.open(image_path).convert("RGB")
    img    = img.resize((image_size, image_size), Image.BICUBIC)
    img_np = np.array(img) / 255.0

    lab    = rgb2lab(img_np).astype("float32")
    L      = (lab[:, :, 0] / 50.0) - 1.0
    L_tensor = torch.tensor(L).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        AB_pred = G(L_tensor).cpu().squeeze(0)

    L_np             = (L_tensor.cpu().squeeze().numpy() + 1.0) * 50.0
    AB_np            = AB_pred.permute(1, 2, 0).numpy() * 128.0

    lab_out          = np.zeros((image_size, image_size, 3), dtype="float32")
    lab_out[:, :, 0] = L_np
    lab_out[:, :, 1:]= AB_np

    rgb_colorized    = np.clip(lab2rgb(lab_out), 0, 1)
    gray             = np.clip(np.stack([L_np / 100.0] * 3, axis=-1), 0, 1)

    return gray, rgb_colorized


def convert_to_grayscale(image_path, image_size=256):
    """Convert color image to grayscale for input"""
    img    = Image.open(image_path).convert("RGB")
    img    = img.resize((image_size, image_size), Image.BICUBIC)
    img_np = np.array(img) / 255.0
    lab    = rgb2lab(img_np).astype("float32")
    L_np   = lab[:, :, 0] / 100.0
    gray   = np.clip(np.stack([L_np] * 3, axis=-1), 0, 1)
    return gray

In [5]:
import random

# Get all val images
all_files = sorted([
    os.path.join(VAL_PATH, f)
    for f in os.listdir(VAL_PATH)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

# Randomly select 15
random.seed(42)    # fixed seed for reproducibility
selected = random.sample(all_files, 15)

print(f"Selected {len(selected)} images")
for i, f in enumerate(selected):
    print(f"  {i+1:2d}. {os.path.basename(f)}")

Selected 15 images
   1. 000000107554.jpg
   2. 000000021167.jpg
   3. 000000260657.jpg
   4. 000000230166.jpg
   5. 000000210299.jpg
   6. 000000134322.jpg
   7. 000000098520.jpg
   8. 000000517687.jpg
   9. 000000083531.jpg
  10. 000000563470.jpg
  11. 000000400573.jpg
  12. 000000027620.jpg
  13. 000000025228.jpg
  14. 000000089648.jpg
  15. 000000205647.jpg


In [ ]:
results = []

for i, image_path in enumerate(selected):
    gray, colorized = colorize(image_path, G, DEVICE, IMAGE_SIZE)
    results.append((gray, colorized, os.path.basename(image_path)))
    print(f"Processed {i+1:2d}/15 — {os.path.basename(image_path)}")

print(f"\nAll 15 images processed!")

In [7]:
fig, axes = plt.subplots(15, 2, figsize=(8, 15 * 3))
fig.suptitle(
    "Image Colorization Results\nGrayscale Input (Left) vs Colorized Output (Right)",
    fontsize=14, fontweight="bold", y=1.001
)

for i, (gray, colorized, filename) in enumerate(results):
    axes[i, 0].imshow(gray, cmap="gray")
    axes[i, 0].set_title(f"Grayscale {i+1}", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(colorized)
    axes[i, 1].set_title(f"Colorized {i+1}", fontsize=9)
    axes[i, 1].axis("off")

plt.tight_layout()
save_path = os.path.join(OUTPUT_PATH, "showcase_15_images.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Showcase saved to {save_path}")

Showcase saved to /home/61447093s/colorizer_gan/outputs/showcase/showcase_15_images.png


In [8]:
# Split into 3 pages of 5 images each — easier to read in report
for page in range(3):
    start = page * 5
    end   = start + 5
    batch = results[start:end]

    fig, axes = plt.subplots(5, 2, figsize=(8, 20))
    fig.suptitle(
        f"Image Colorization Results — Page {page+1}/3\n"
        f"Grayscale Input (Left) vs Colorized Output (Right)",
        fontsize=13, fontweight="bold"
    )

    for i, (gray, colorized, filename) in enumerate(batch):
        axes[i, 0].imshow(gray, cmap="gray")
        axes[i, 0].set_title(f"Image {start+i+1} — Grayscale Input", fontsize=9)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(colorized)
        axes[i, 1].set_title(f"Image {start+i+1} — Colorized Output", fontsize=9)
        axes[i, 1].axis("off")

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_PATH, f"showcase_page_{page+1}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Page {page+1} saved to {save_path}")

Page 1 saved to /home/61447093s/colorizer_gan/outputs/showcase/showcase_page_1.png
Page 2 saved to /home/61447093s/colorizer_gan/outputs/showcase/showcase_page_2.png
Page 3 saved to /home/61447093s/colorizer_gan/outputs/showcase/showcase_page_3.png
